In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:


import os
import pandas as pd

path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(path)

print(f"Shape: {df.shape}") # the dataset shape


In [ ]:
# Task 2: Write your code here:

df.head() # print firstt five samples

In [ ]:
# Task 3: Write your code here:
df.info() # to see the types of our features for encoding and missing values (later)

In [ ]:
# Task 4: Write your code here:
df.describe() # for scaling (later)

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt # import matplotlib for visualization

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

d= df.drop(columns=['Order_ID'])
d

In [ ]:
# Task 2: Write your code here:

# 1) lets see our missing values first and how many they are if not alot then we will not drop the features because they might be relevant or beneficial to the models predictions

print("Missing values:")
print(d .isnull().sum())



In [ ]:
# 2) # hard coded each feature that had missing values and i thought filling it wiht the mode would be the best option sisnce feature include both catagorical and numrical data

d['Weather'] = d['Weather'].fillna(d['Weather'].mode()[0])

d['Traffic_Level'] = d['Traffic_Level'].fillna(d['Traffic_Level'].mode()[0])

d['Courier_Experience_yrs'] = d['Courier_Experience_yrs'].fillna(d['Courier_Experience_yrs'].mode()[0])
d['Time_of_Day'] = d['Time_of_Day'].fillna(d['Time_of_Day'].mode()[0])
d['Delivery_Time'] = d['Delivery_Time'].fillna(d['Delivery_Time'].mode()[0])

print("Missing values remaining:", d.isnull().sum().sum())


In [ ]:
# Task 3: Write your code here:

def check_duplicates(d):
  duplicates = d.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    d.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(d)


In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Labeleencoder used

cols = ['Weather','Traffic_Level','Time_of_Day','Vehicle_Type']


for col in cols:
    le = LabelEncoder()
    d[col] = le.fit_transform(d[col].astype(str))

d.head()

# ONE HOT ENCODER CODE

# print('data before encoding:\n', categories) #show before encoding

# onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
# data_onehot_encoded = onehot_encoder.fit_transform(categories) # Apply fit_transform to the copied

# print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET # IMP

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:




In [ ]:
# Task 6: Write your code here:

 # i believe its imbalance from this line of code and the distribution perfomed earlier

target_column =d["Delivery_Time"]

def check_target_imbalance(d, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:

#
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)



In [ ]:
# Task 2,3,4,5: Write your code here:



# 2) i will use k fold since the dataset is small and no need to use stratified since there is no imbalance


from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1): # we only take the indices of x because y is corresponding with it
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)


# 3)  # import need libraries for ou model and evaluation

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
print("Model trained!")

# 4)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")

# 5)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))


mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")





In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:



In [ ]:
# Task Bonus: Write your code here: